In [ ]:
"""
Spatio-Temporal Graph Neural Network (ST-GNN) for BRT Demand Forecasting
-------------------------------------------------------------------------
This file contains the complete end-to-end implementation of the Dual-Layer 
GCN + LSTM architecture designed for a 30-station linear transit corridor.

Requirements:
    pip install torch numpy scikit-learn
"""

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import math

# ==========================================
# 1. CONFIGURATION & HYPERPARAMETERS
# ==========================================
NUM_STATIONS = 30          # N = 30 BRT Stations
SEQ_LENGTH = 6             # T = 6 past hours
HIDDEN_GCN = 64            # Hidden dimensions for GCN layers
HIDDEN_LSTM = 128          # Hidden dimensions for LSTM
BATCH_SIZE = 32            # Training batch size
EPOCHS = 50                # Number of training epochs
LEARNING_RATE = 0.001      # Adam optimizer learning rate
DROPOUT_RATE = 0.2         # Dropout to prevent overfitting

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# ==========================================
# 2. GRAPH CONSTRUCTION (Adjacency Matrix)
# ==========================================
def build_normalized_adjacency(n_stations):
    """
    Constructs the linear corridor graph, adds self-loops, and applies 
    the mathematically normalized Laplacian transformation.
    """
    # 1. Initialize linear bidirectional adjacency matrix
    A = np.zeros((n_stations, n_stations), dtype=np.float32)
    for i in range(n_stations):
        if i > 0: A[i, i-1] = 1.0  # Edge to previous station
        if i < n_stations - 1: A[i, i+1] = 1.0  # Edge to next station
        
    # 2. Add self-loops (A + I_N)
    A_tilde = A + np.eye(n_stations, dtype=np.float32)
    
    # 3. Compute Degree Matrix D_tilde and D^(-1/2)
    D_tilde = np.sum(A_tilde, axis=1)
    D_inv_sqrt = np.power(D_tilde, -0.5)
    D_inv_sqrt[np.isinf(D_inv_sqrt)] = 0.0
    D_mat_inv_sqrt = np.diag(D_inv_sqrt)
    
    # 4. Normalized Adjacency: D^(-1/2) * A_tilde * D^(-1/2)
    A_norm = np.dot(np.dot(D_mat_inv_sqrt, A_tilde), D_mat_inv_sqrt)
    
    return torch.tensor(A_norm, dtype=torch.float32)

# ==========================================
# 3. SYNTHETIC DATA GENERATOR & SCALER
# ==========================================
def generate_brt_data(num_samples=2000, n_stations=30):
    """
    Generates synthetic high-variance passenger demand data replicating 
    morning/evening commuting cycles across a localized transit corridor.
    """
    time = np.arange(num_samples)
    data = np.zeros((num_samples, n_stations), dtype=np.float32)
    
    for i in range(n_stations):
        # Base daily cycle (sine wave) + localized noise
        base_cycle = (np.sin(time * (2 * np.pi / 24)) + 1) * 1000 
        station_variance = np.random.normal(0, 300, num_samples)
        # Create multi-hop ripple effects (lagged surges from previous stations)
        ripple = data[:, i-1] * 0.3 if i > 0 else 0
        
        data[:, i] = np.clip(base_cycle + station_variance + ripple, 0, None)
        
    return data

def create_sequences(data, seq_length):
    """Slides a temporal window of size T across the dataset."""
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i : i + seq_length])
        y.append(data[i + seq_length])
    return np.array(X), np.array(y)

# Min-Max Scaler
class MinMaxDataScaler:
    def __init__(self):
        self.min_val = None
        self.max_val = None

    def fit_transform(self, data):
        self.min_val = np.min(data)
        self.max_val = np.max(data)
        return (data - self.min_val) / (self.max_val - self.min_val)

    def inverse_transform(self, data_scaled):
        return data_scaled * (self.max_val - self.min_val) + self.min_val

# ==========================================
# 4. NEURAL NETWORK MODULES
# ==========================================
class GraphConvolution(nn.Module):
    """A single spatial Message Passing layer (1-hop)."""
    def __init__(self, in_features, out_features):
        super(GraphConvolution, self).__init__()
        self.weight = nn.Parameter(torch.FloatTensor(in_features, out_features))
        self.bias = nn.Parameter(torch.FloatTensor(out_features))
        self.reset_parameters()

    def reset_parameters(self):
        stdv = 1. / math.sqrt(self.weight.size(1))
        self.weight.data.uniform_(-stdv, stdv)
        self.bias.data.uniform_(-stdv, stdv)

    def forward(self, x, adj):
        # x: (Batch, Stations, Features)
        support = torch.matmul(x, self.weight)
        # adj: (Stations, Stations)
        output = torch.matmul(adj, support) + self.bias
        return output

class ST_GNN(nn.Module):
    """
    Proposed Architecture: Dual-Layer GCN + LSTM
    """
    def __init__(self, n_stations, seq_length, gcn_hidden, lstm_hidden, dropout):
        super(ST_GNN, self).__init__()
        self.n_stations = n_stations
        
        # Spatial Module: Dual-Layer GCN
        self.gcn1 = GraphConvolution(in_features=1, out_features=gcn_hidden)
        self.gcn2 = GraphConvolution(in_features=gcn_hidden, out_features=gcn_hidden)
        
        # Temporal Module: LSTM
        # Input to LSTM is the flattened spatial features of the whole graph
        self.lstm_input_dim = n_stations * gcn_hidden
        self.lstm = nn.LSTM(input_size=self.lstm_input_dim, 
                            hidden_size=lstm_hidden, 
                            batch_first=True)
        
        # Output Module
        self.dropout = nn.Dropout(dropout)
        self.linear = nn.Linear(lstm_hidden, n_stations)
        self.relu = nn.ReLU()

    def forward(self, x, adj):
        # x shape: (Batch, Seq_Length, Stations)
        batch_size = x.size(0)
        seq_len = x.size(1)
        
        # Reshape to (Batch, Seq_Length, Stations, 1) for GCN features
        x = x.unsqueeze(-1)
        
        spatial_outputs = []
        for t in range(seq_len):
            # Extract traffic state at time step t
            x_t = x[:, t, :, :]
            
            # Layer 1: 1-hop spatial dependencies
            h1 = self.relu(self.gcn1(x_t, adj))
            h1 = self.dropout(h1)
            
            # Layer 2: 2-hop downstream ripple effects
            h2 = self.relu(self.gcn2(h1, adj))
            
            # Flatten spatial graph to feed into sequence model
            spatial_outputs.append(h2.view(batch_size, -1))
            
        # Stack temporal states: (Batch, Seq_Length, Flattened_Graph)
        lstm_in = torch.stack(spatial_outputs, dim=1)
        
        # Pass sequence through LSTM
        lstm_out, (h_n, c_n) = self.lstm(lstm_in)
        
        # Take the hidden state of the final time step
        final_temporal_state = lstm_out[:, -1, :]
        
        # Map back to passenger counts for each station
        predictions = self.linear(final_temporal_state)
        return predictions

# ==========================================
# 5. EXECUTION PIPELINE
# ==========================================
def main():
    print("1. Generating BRT Data and Constructing Spatial Graph...")
    adj_matrix = build_normalized_adjacency(NUM_STATIONS)
    raw_data = generate_brt_data(num_samples=2000, n_stations=NUM_STATIONS)
    
    # Scale Data
    scaler = MinMaxDataScaler()
    scaled_data = scaler.fit_transform(raw_data)
    
    # Create Sequences (X) and Targets (Y)
    X, y = create_sequences(scaled_data, SEQ_LENGTH)
    
    # Split: 70% Train, 15% Val, 15% Test
    n = len(X)
    train_end = int(n * 0.7)
    val_end = int(n * 0.85)
    
    X_train, y_train = torch.tensor(X[:train_end], dtype=torch.float32), torch.tensor(y[:train_end], dtype=torch.float32)
    X_test, y_test = torch.tensor(X[val_end:], dtype=torch.float32), torch.tensor(y[val_end:], dtype=torch.float32)
    
    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
    
    # Initialize Model, Optimizer, and Loss
    print("2. Initializing Proposed ST-GNN Architecture...")
    model = ST_GNN(n_stations=NUM_STATIONS, seq_length=SEQ_LENGTH, 
                   gcn_hidden=HIDDEN_GCN, lstm_hidden=HIDDEN_LSTM, dropout=DROPOUT_RATE)
    
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    
    print("3. Starting Training Loop...")
    for epoch in range(1, EPOCHS + 1):
        model.train()
        epoch_loss = 0
        for batch_x, batch_y in train_loader:
            optimizer.zero_grad()
            predictions = model(batch_x, adj_matrix)
            loss = criterion(predictions, batch_y)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            
        if epoch % 10 == 0:
            print(f"Epoch {epoch:02d}/{EPOCHS} | Train Loss (MSE): {epoch_loss/len(train_loader):.4f}")

    # ==========================================
    # 6. EVALUATION METRICS
    # ==========================================
    print("\n4. Evaluating Model on 15% Holdout Test Set...")
    model.eval()
    with torch.no_grad():
        test_preds_scaled = model(X_test, adj_matrix).numpy()
        y_test_scaled = y_test.numpy()
        
        # Inverse transform to get actual passenger counts
        test_preds = scaler.inverse_transform(test_preds_scaled)
        ground_truth = scaler.inverse_transform(y_test_scaled)
        
        # Calculate Metrics
        mae = mean_absolute_error(ground_truth, test_preds)
        rmse = np.sqrt(mean_squared_error(ground_truth, test_preds))
        r2 = r2_score(ground_truth, test_preds)
        
        # Calculate MAPE safely (avoid division by zero)
        mape = np.mean(np.abs((ground_truth - test_preds) / (ground_truth + 1e-5))) * 100
        
        print("-" * 40)
        print("FINAL ST-GNN TEST RESULTS")
        print("-" * 40)
        print(f"MAE:       {mae:.2f}")
        print(f"RMSE:      {rmse:.2f}")
        print(f"MAPE:      {mape:.2f}%")
        print(f"R-Squared: {r2:.4f}")
        print("-" * 40)

if __name__ == "__main__":
    main()